# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Users/albertoreydelafe@gmail.com/semana01-agentops"  # experimento dentro de mi carpeta personal, que ya existe
mlflow.set_experiment(EXPERIMENT_NAME)
#mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})
# quito el mlflow.set_tags de aqui porque fuera de un run abre un run nuevo y luego falla el start_run de la seccion 4

2026/10/02 19:55:11 INFO mlflow.tracking.fluent: Experiment with name '/Users/albertoreydelafe@gmail.com/semana01-agentops' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3272606949732393', creation_time=1790970911902, effective_trace_archival_retention=None, experiment_id='3272606949732393', last_update_time=1790970911902, lifecycle_stage='active', name='/Users/albertoreydelafe@gmail.com/semana01-agentops', tags={'mlflow.experiment.sourceName': '/Users/albertoreydelafe@gmail.com/semana01-agentops',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'albertoreydelafe@gmail.com',
 'mlflow.ownerId': '70425375825392'}, trace_location=None, workspace='default'>

## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.


In [0]:
@mlflow.trace
def route_question(question: str) -> str:
    text = question.lower()
    # TODO 1: esta función decide de qué trata la pregunta mirando si trae ciertas palabras. Primero miro si habla de seguridad (token, secreto, contraseña,  privacidad), porque si alguien menciona un secreto es lo más importante y hay que avisarle antes que nada
    # Si no, miro si habla de producción (producción, api, servicio).Si no cuadra en ninguna, la mando a "mlflow", que es la ruta por defecto, así siempre hay una respuesta.El orden importa ya que si la pregunta tiene palabras de dos tema gana el que compruebo antes
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/10/02 19:55:37 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o11.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-c9dc44347cd459ce5f6133dbd7b01aa9), Trace(trace_id=tr-3ba77def1f8d1f907f37815ccb557ce3), Trace(trace_id=tr-f9863e56935987b1a1ff9e6c8e1298a5)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.

RESPUESTA:
En la traza inspeccionada,la pregunta "¿Qué objeto agrupa varios runs?".
La ruta: mlflow (no tenía palabras de seguridad ni de producción, así que fue a la ruta por defecto).
El contexto que se recupero fue: "Un experimento agrupa runs con parámetros, métricas y artefactos."
Si la ruta fuera incorrecta miraría en la traza qué devolvió route_question y cambiaría las palabras clave o el orden de los if.


## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.




In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    # TODO 2: añade dos casos más y comprueba que siguen pasando.
    #Añado una de secretos y otra de producción.
    # Cada una lleva la palabra que espero ver en la respuesta. Si el agente la manda a la ruta correcta, la palabra aparece y el caso pasa
    {"question": "¿Qué pasa con la privacidad de los datos?", "must_include": "secretos"},  # lleva "privacidad", así que va a la ruta de seguridad
    {"question": "¿Qué necesito para poner un modelo en producción?", "must_include": "monitorización"},  # lleva "producción", así que va a la ruta de producción
]


evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Qué pasa con la privacidad de los datos?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué necesito para poner un modelo en producción?,monitorización,"Ruta: production. En producción importan contratos, versionado y monitorización.",true


[Trace(trace_id=tr-62010665b7c19360fb3526b717f0a9de), Trace(trace_id=tr-c3f76823565f389b16790b4cef9a6f67), Trace(trace_id=tr-5332dc8eb728521c93acd3f32b710dbc), Trace(trace_id=tr-0058203be4479db202f740567d5463ac), Trace(trace_id=tr-8a6fa3e4ecec3221814bd2ee0bb10604)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.


In [0]:
# TODO 3: guardo la evaluación en un run de MLflow. Si no el resultado solo se ve en pantalla y se pierde
# Apunto cuántos casos he probado (case_count) y qué porcentaje ha pasado (pass_rate: 1.0 significa que pasan todos),y guardo el detalle de cada caso en un fichero (results.json) dentro del run

with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/10/02 20:07:22 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o19.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

{'evaluation_run_id': '6d9500c6ad194e4899672a3c96ac8a4f', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [0]:
# TODO 4: le pido a MLflow que me enseñe solo los runs de mi agente actual, el "v1-deterministic".Lo hago con la etiqueta agent.version, que puse al guardar cada run.Así encuentro mi evaluación entre los runs que tenga del experimento
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
6d9500c6ad194e4899672a3c96ac8a4f,v1-deterministic,1.0


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.


EVALUATION_RUN_ID: 6d9500c6ad194e4899672a3c96ac8a4f (lo imprime la celda de la sección 4, `run.info.run_id` y lo recupero con `search_runs`)

pass_rate: 1.0 (lo calculo en la sección 4 como la media de los casos que pasan, `evaluation_df["passed"].mean()`)

#Respuesta a : qué riesgo cubrirías en la siguiente versión del agente.

Mi agente solo busca palabras clave, así que si le preguntan algo que no tiene que ver con el curso, igual responde con la información de MLflow como si supiera. En la siguiente versión añadiría una ruta que diga 'no puedo responder a eso' cuando no reconozca la pregunta.

- Captura de la traza(trace ID: tr-c9dc44347cd459ce5f6133dbd7b01aa9), al lado de status:ok

![traza_1790972444743.png](./traza_1790972444743.png "traza_1790972444743.png")